# From a source to an Artefact

This notebook follows a read step by step, from the document you give to LODE to the `Artefact` it returns. It is for whoever writes a new format (a profile): at each step it shows what to look at to know whether the mapper does what you expect.

You will declare a profile with an empty mapping, read a document with it, and compare the result with the two readers LODE ships, SKOS and OWL. What to write for a real format is in `docs/extending.md`; this page is about **inspecting** what happens.

The imports from `lode.reader` below are for people who extend LODE. They are not part of the stable API of `docs/usage/python_api.md`.

## 0. The input

A small SKOS vocabulary: one scheme and two concepts. The concept `ex:citrus` also has a triple, `ex:origin`, that the SKOS mapper does not know. We will find it again at the end.

The reader takes a path or a URL, so the Turtle goes through a temporary file.

In [1]:
import tempfile
from pathlib import Path

turtle = """
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .
@prefix ex:   <http://example.org/fruit#> .

ex:scheme a skos:ConceptScheme ;
    skos:prefLabel "Fruit"@en ;
    skos:hasTopConcept ex:fruit .

ex:fruit a skos:Concept ;
    skos:inScheme ex:scheme ;
    skos:topConceptOf ex:scheme ;
    skos:prefLabel "Fruit"@en ;
    skos:definition "Edible product of a plant."@en .

ex:citrus a skos:Concept ;
    skos:inScheme ex:scheme ;
    skos:prefLabel "Citrus"@en ;
    skos:altLabel "Citrus fruit"@en ;
    skos:broader ex:fruit ;
    ex:origin "Asia" .
"""

path = Path(tempfile.mkdtemp(prefix="lode_")) / "fruit.ttl"
path.write_text(turtle, encoding="utf-8")

557

## 1. Load the graph

The loader turns the document into an `rdflib` graph. It also returns the IRI of the documented artefact and, when imports are requested, the imported documents.

Before writing a mapper, look at the `rdf:type` values the document uses: they are the classes the mapper has to translate.

In [2]:
from collections import Counter

from rdflib import RDF

from lode.reader.pipeline import load_graph

graph, artefact_iri, imported_uris = load_graph(str(path))

print(type(graph).__name__, len(graph), "triples")
print("artefact_iri:", artefact_iri, "| imports:", imported_uris)
print(Counter(str(o).rsplit("#", 1)[-1] for o in graph.objects(None, RDF.type)))

Graph 14 triples
artefact_iri: http://example.org/fruit#scheme | imports: set()
Counter({'Concept': 2, 'ConceptScheme': 1})


## 2. Declare an empty profile

A format is a **profile**: a package that tells LODE how to read it. Importing the package registers the profile. We write the smallest possible one, with three files:

| File | Role |
|------|------|
| `__init__.py` | Declares the `Profile`: its name and where its reader, viewer and `Artefact` class are. |
| `artefact.py` | The `Artefact` subclass users read the format with. It carries no code. |
| `reader.yaml` | The mapper. Here it is empty: it is merged on top of `lode/reader/config/base.yaml`, so an empty mapper reads like the base. |

The profile has no reader of its own: it points at `Reader`, which implements the six phases. A real format adds a reader class only for what the YAML cannot say.

It is declared with `enabled=False`, as `docs/extending.md` advises while a format is under development. A disabled profile is refused when it is named by a string (`lode.read(path, "empty")` fails), so it is read through its class.

In [3]:
import sys
import textwrap

package = Path(tempfile.mkdtemp(prefix="lode_profile_")) / "empty_profile"
package.mkdir()

# the profile: a name, and where the reader, the viewer and the Artefact class are
(package / "__init__.py").write_text(textwrap.dedent('''
    """An empty profile: it maps nothing, so it reads like the base."""
    from lode.profiles import Profile, register

    register(Profile(
        name="empty",
        package=__name__,
        reader="lode.reader.reader:Reader",   # no reader of its own
        viewer="lode.viewer.base_viewer:BaseViewer",       # no viewer yet
        artefact=".artefact:EmptyArtefact",
        enabled=False,                                      # under development
    ))
'''), encoding="utf-8")

# the Artefact class users read the format with
(package / "artefact.py").write_text(textwrap.dedent('''
    """EmptyArtefact: the Artefact class of the empty profile."""
    from lode.artefact import Artefact


    class EmptyArtefact(Artefact):
        """An artefact read with the empty profile."""
'''), encoding="utf-8")

# the mapper: empty, so everything comes from base.yaml
(package / "reader.yaml").write_text("mapper: {}\nenricher: {}\n", encoding="utf-8")

sys.path.insert(0, str(package.parent))
import empty_profile  # importing the package registers the profile

from lode import profiles

print(profiles.get("empty"))

Profile(name='empty', package='empty_profile', reader='lode.reader.reader:Reader', viewer='lode.viewer.base_viewer:BaseViewer', artefact='.artefact:EmptyArtefact', post_checks=None, enabled=False)


## 3. Read with it

The profile is registered, so its `Artefact` class can read a document. The result is what the base alone makes of it:

In [4]:
from empty_profile.artefact import EmptyArtefact

empty = EmptyArtefact(str(path))

print(empty.read_as, empty.artefact_iri)
print(Counter(type(i).__name__ for i in empty.instances()))

empty http://example.org/fruit#scheme
Counter({'Statement': 11, 'Property': 8, 'Resource': 3, 'Concept': 2})


Nothing of SKOS is mapped: every subject became a `Resource`, and every triple that nobody consumed is a `Statement`. The two `Concept` instances are not the concepts of the vocabulary: they are `skos:Concept` and `skos:ConceptScheme`, created as values of `rdf:type`, which the base maps to `has_type`.

## 4. The six phases

`lode.read` and the `Artefact` classes run six phases over the graph. All of them fill one cache, a dictionary `{IRI: set of instances}` (a set, because one IRI can be used for more than one kind of entity). A profile can extend or replace a phase.

| Phase | What it does |
|-------|--------------|
| 1 `phase1_classify_from_predicates` | Creates instances for subjects whose predicates reveal their class, even without `rdf:type`. |
| 2 `phase2_create_from_types` | Creates instances from the `rdf:type` declarations the mapper knows. |
| 3 `phase3_populate_properties` | Fills each instance from its triples, calling the setters or handlers the mapper names. |
| 4 `phase4_process_group_axioms` | Processes axioms that involve several entities at once (OWL). |
| 5 `phase5_fallback` | Gives a default class to the IRIs no earlier phase classified. |
| 6 `phase6_create_statements` | Keeps every triple no phase consumed as a `Statement`. |

Here we run them one at a time with the empty profile and count the instances by class after each one.

In [5]:
from lode.reader.config_manager import ConfigManager

PHASES = ["phase1_classify_from_predicates", "phase2_create_from_types",
          "phase3_populate_properties", "phase4_process_group_axioms",
          "phase5_fallback", "phase6_create_statements"]


def run_phases(reader, cache):
    """Run the six phases one at a time and count the instances by class after each one."""
    for name in PHASES:
        getattr(reader, name)()
        counts = Counter(type(i).__name__ for group in cache.values() for i in group)
        print(f"{name:34} {dict(counts)}")


profile = profiles.get("empty")
cache = {}
reader = profile.make_reader(graph, cache, ConfigManager(profile))

run_phases(reader, cache)

phase1_classify_from_predicates    {}
phase2_create_from_types           {}
phase3_populate_properties         {}
phase4_process_group_axioms        {}
phase5_fallback                    {'Resource': 3, 'Concept': 2}
phase6_create_statements           {'Resource': 3, 'Concept': 2, 'Property': 8, 'Statement': 11, 'Literal': 5}


Read the counts like this:

- Phases 1 to 4 create nothing: the base maps no SKOS type.
- Phase 5 gives the default class, `Resource`, to every subject of the document.
- Phase 6 keeps as a `Statement` every triple that nobody consumed. Only the `rdf:type` triples are consumed.

The cache is keyed by IRI. Literals are in it too, under keys of their own.

In [6]:
for key, group in list(cache.items())[:4]:
    print(key, "->", group)

http://example.org/fruit#scheme -> {Resource('http://example.org/fruit#scheme')}
http://www.w3.org/2004/02/skos/core#ConceptScheme -> {Concept('http://www.w3.org/2004/02/skos/core#ConceptScheme')}
http://example.org/fruit#citrus -> {Resource('http://example.org/fruit#citrus')}
http://www.w3.org/2004/02/skos/core#Concept -> {Concept('http://www.w3.org/2004/02/skos/core#Concept')}


## 5. How an entity gets filled

The counts say how many entities exist, not what is in them. Here we follow one, `ex:citrus`, with the SKOS profile: first right after phase 2, then after phase 3. An entity prints with its IRI, and `filled` keeps only the fields that hold a value (`vars` lists the fields of a model object, as in section 2.3 of the first notebook).

In [7]:
from pprint import pprint

from rdflib import URIRef
from rdflib.namespace import SKOS

skos = profiles.get("skos")
skos_config = ConfigManager(skos)
skos_cache = {}
skos_reader = skos.make_reader(graph, skos_cache, skos_config)

citrus_iri = URIRef("http://example.org/fruit#citrus")
fruit_iri = URIRef("http://example.org/fruit#fruit")


def filled(entity):
    """Return the fields of an entity that hold a value."""
    return {name: value for name, value in vars(entity).items() if value}


skos_reader.phase1_classify_from_predicates()
skos_reader.phase2_create_from_types()

for entity in skos_cache[citrus_iri]:
    print("after phase 2:", entity)
    pprint(filled(entity))

after phase 2: Concept('http://example.org/fruit#citrus')
{'has_identifier': 'http://example.org/fruit#citrus'}


After phase 2 the entity exists, with its class and its IRI, and nothing else. Phase 3 reads its triples and fills it:

In [8]:
skos_reader.phase3_populate_properties()

for entity in skos_cache[citrus_iri]:
    print("after phase 3:", entity)
    pprint(filled(entity))

after phase 3: Concept('http://example.org/fruit#citrus')
{'has_alternative_label': [Literal('Citrus fruit', lang='en')],
 'has_identifier': 'http://example.org/fruit#citrus',
 'has_preferred_label': [Literal('Citrus', lang='en')],
 'is_included_in': [Model('http://example.org/fruit#scheme')],
 'is_sub_concept_of': [Concept('http://example.org/fruit#fruit')]}


Each triple of `ex:citrus` became a value in a field. The mapper says which one: for every predicate it names a **setter**, and the setter `set_x` fills the field `x`.

In [9]:
mapping = skos_config.get_property_mapping()

for predicate in (SKOS.prefLabel, SKOS.altLabel, SKOS.inScheme, SKOS.broader):
    print(predicate.n3(graph.namespace_manager), "->", mapping[predicate]["setters"])

skos:prefLabel -> [{'set_has_preferred_label': 'Literal'}]
skos:altLabel -> [{'set_has_alternative_label': 'Literal'}]
skos:inScheme -> [{'set_is_included_in': <class 'lode.models.model.Model'>}]
skos:broader -> [{'set_is_sub_concept_of': {'targets': <class 'lode.models.concept.Concept'>, 'inverse_of': 'set_is_super_concept_of'}}]


Read the setters like this:

- `'Literal'`: the object of the triple becomes a `Literal` (with its language) and is added to the field.
- A class (`Model`): the object is an IRI, resolved to the instance of that class, so `skos:inScheme ex:scheme` fills `is_included_in` with the `Model`.
- `inverse_of`: the setter on the other side is called too. Only `skos:broader` is asserted between `ex:citrus` and `ex:fruit`, yet the field of `ex:fruit` is filled as well:

In [10]:
for entity in skos_cache[fruit_iri]:
    print(entity, "->", entity.get_is_super_concept_of())

Concept('http://example.org/fruit#fruit') -> [Concept('http://example.org/fruit#citrus')]


## 6. Same document, two other readers

Now read the same document with the two formats LODE ships. The profile decides what the document becomes.

In [11]:
from lode import OwlOntology, SkosVocabulary
from lode.models import Statement

for artefact_class in (EmptyArtefact, SkosVocabulary, OwlOntology):
    artefact = artefact_class(str(path))
    counts = Counter(type(i).__name__ for i in artefact.instances())
    print(f"{artefact_class.__name__:15} {len(artefact.instances(Statement)):2} statements | {dict(counts)}")

EmptyArtefact   11 statements | {'Resource': 3, 'Concept': 2, 'Property': 8, 'Statement': 11}


SkosVocabulary   1 statements | {'Concept': 2, 'Model': 1, 'Property': 1, 'Statement': 1}


OwlOntology     11 statements | {'Model': 1, 'Individual': 2, 'Concept': 3, 'Datatype': 1, 'Annotation': 8, 'Statement': 11}


The counts say how much each profile understood. To see **what** it made of the document, here are all the instances each one produced, with the fields that hold a value (`filled` comes from section 5), as a dictionary `{entity: fields}` printed with `pprint`. The `Statement` instances are shown on one line, as subject, predicate, object. `has_namespaces`, the prefixes of the document, is left out.

In [12]:
for artefact_class in (EmptyArtefact, SkosVocabulary, OwlOntology):
    artefact = artefact_class(str(path))
    print(f"\n == {artefact_class.__name__} (read_as={artefact.read_as})")

    entities = [i for i in artefact.instances()]
    pprint({entity for entity in entities}, width=120, sort_dicts=False)


 == EmptyArtefact (read_as=empty)


{Concept('http://www.w3.org/2004/02/skos/core#Concept'),
 Concept('http://www.w3.org/2004/02/skos/core#ConceptScheme'),
 Property('http://www.w3.org/2004/02/skos/core#topConceptOf'),
 Property('http://www.w3.org/2004/02/skos/core#hasTopConcept'),
 Property('http://www.w3.org/2004/02/skos/core#broader'),
 Property('http://www.w3.org/2004/02/skos/core#inScheme'),
 Property('http://www.w3.org/2004/02/skos/core#prefLabel'),
 Property('http://www.w3.org/2004/02/skos/core#definition'),
 Property('http://www.w3.org/2004/02/skos/core#altLabel'),
 Property('http://example.org/fruit#origin'),
 Resource('http://example.org/fruit#scheme'),
 Resource('http://example.org/fruit#citrus'),
 Resource('http://example.org/fruit#fruit'),
 Statement('N1c6bbf91a5eb4cb9abae29c62a5482dd'),
 Statement('N1856a582a89d413ab2feea4e73070dcc'),
 Statement('N2e681fe9d98e4bfeb6611c993e8391a4'),
 Statement('N51ad1a4e5f0243b997dfb9133a4fbfae'),
 Statement('Nd21fb93f2ffe4c648e2039081b11f8bf'),
 Statement('Na3898477318340


 == SkosVocabulary (read_as=skos)
{Concept('http://example.org/fruit#citrus'),
 Concept('http://example.org/fruit#fruit'),
 Model('http://example.org/fruit#scheme'),
 Property('http://example.org/fruit#origin'),
 Statement('Ndde331022fba41a38573d5ef625e5474')}



 == OwlOntology (read_as=owl)
{Annotation('http://www.w3.org/2004/02/skos/core#inScheme'),
 Annotation('http://www.w3.org/2004/02/skos/core#definition'),
 Annotation('http://www.w3.org/2004/02/skos/core#prefLabel'),
 Annotation('http://www.w3.org/2004/02/skos/core#altLabel'),
 Annotation('http://www.w3.org/2004/02/skos/core#topConceptOf'),
 Annotation('http://www.w3.org/2004/02/skos/core#hasTopConcept'),
 Annotation('http://www.w3.org/2004/02/skos/core#broader'),
 Annotation('http://example.org/fruit#origin'),
 Concept('http://www.w3.org/2004/02/skos/core#Concept'),
 Concept('http://www.w3.org/2004/02/skos/core#ConceptScheme'),
 Concept('http://www.w3.org/2002/07/owl#Thing'),
 Datatype('http://www.w3.org/2000/01/rdf-schema#Literal'),
 Individual('http://example.org/fruit#citrus'),
 Individual('http://example.org/fruit#fruit'),
 Model('http://example.org/fruit#scheme'),
 Statement('N8dacee5e46f743b399818bc3daba50bd'),
 Statement('N8bd66deb456c4fb1b06aad23d92b9518'),
 Statement('N724a2f

The same document gives three different artefacts, and the reader raises no error and no warning: it cannot know that a format is wrong for the document.

- With the **SKOS** profile the vocabulary is recognised: each `Concept` has its labels, its scheme (`is_included_in`) and its place in the hierarchy (`is_sub_concept_of`, and `is_super_concept_of` on the other side), and the scheme is a `Model` that knows its top concept. Only the triple SKOS does not know, `ex:origin`, is left as a `Statement`.
- With the **OWL** profile `skos:Concept` is just a class like any other, so the concepts of the vocabulary become `Individual` instances of it (`has_type`), and every SKOS predicate becomes an `Annotation`, used by a `Statement`. The profile also creates two entities of its own, `owl:Thing` and `rdfs:Literal`, that are not in the document.
- With your **empty** profile nothing is mapped: each subject is a `Resource` that only knows its type, each predicate a `Property`, and all the information is in the `Statement` instances.

The number of `Statement` is the signal to watch when you write a mapper: the fewer there are, the more of the document the mapper understands.

## 7. The Artefact

Whatever the profile, the result is an `Artefact`: the entities, the RDF graph, the IRI of the artefact and the warnings. Nothing after the reader needs to know how it was built. From here on we use the SKOS one.

In [13]:
vocabulary = SkosVocabulary(str(path))

print(type(vocabulary).__name__, vocabulary.read_as, vocabulary.artefact_iri)
print("entities:", len(vocabulary.entities), "| graph:", len(vocabulary.graph), "triples")

# the dictionary the reader filled: IRI -> set of instances (literals sit under keys of their own)
pprint(vocabulary.entities, width=120)

SkosVocabulary skos http://example.org/fruit#scheme
entities: 10 | graph: 14 triples
{'LITERAL::Asia@^^': {Literal('Asia')},
 'LITERAL::Citrus fruit@en^^': {Literal('Citrus fruit', lang='en')},
 'LITERAL::Citrus@en^^': {Literal('Citrus', lang='en')},
 'LITERAL::Edible product of a plant.@en^^': {Literal('Edible product of a plant.', lang='en')},
 'LITERAL::Fruit@en^^': {Literal('Fruit', lang='en')},
 rdflib.term.BNode('N8d3cc1efd8f84489b1dad6bed7f576b2'): {Statement('N8d3cc1efd8f84489b1dad6bed7f576b2')},
 rdflib.term.URIRef('http://example.org/fruit#citrus'): {Concept('http://example.org/fruit#citrus')},
 rdflib.term.URIRef('http://example.org/fruit#fruit'): {Concept('http://example.org/fruit#fruit')},
 rdflib.term.URIRef('http://example.org/fruit#origin'): {Property('http://example.org/fruit#origin')},
 rdflib.term.URIRef('http://example.org/fruit#scheme'): {Model('http://example.org/fruit#scheme')}}


## 8. Triples the mapper did not consume

Every triple that no phase consumed becomes a `Statement`. For a new mapper this is the quickest check: a non-empty list points at the constructs still missing from the YAML.

In [14]:
for statement in vocabulary.instances(Statement):
    print(statement.get_has_predicate(), "->", statement.get_has_object())

Property('http://example.org/fruit#origin') -> Literal('Asia')


## 9. Where an entity comes from

`provenance(entity)` returns the triples one entity was built from. It is the other way to check a mapper: the triples you expect should all be in it, together with the ones no mapper entry consumed (here `ex:origin`).

In [15]:
(citrus,) = vocabulary.get_instance("http://example.org/fruit#citrus")   # one kind of entity, so one instance

print(vocabulary.provenance(citrus).serialize(format="nt"))

<http://example.org/fruit#citrus> <http://www.w3.org/2004/02/skos/core#broader> <http://example.org/fruit#fruit> .
<http://example.org/fruit#citrus> <http://www.w3.org/2004/02/skos/core#altLabel> "Citrus fruit"@en .
<http://example.org/fruit#citrus> <http://example.org/fruit#origin> "Asia" .
<http://example.org/fruit#citrus> <http://www.w3.org/2004/02/skos/core#prefLabel> "Citrus"@en .
<http://example.org/fruit#citrus> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://www.w3.org/2004/02/skos/core#Concept> .
<http://example.org/fruit#citrus> <http://www.w3.org/2004/02/skos/core#inScheme> <http://example.org/fruit#scheme> .



## Next

What to write for a real format: `docs/extending.md`. The reader itself (the mapper, the reader class, how to change a phase) comes in the next notebook.